In [1]:
import pandas as pd
import spiceypy as spy
import numpy as np
import rebound as rb
from Utils import *
from astroquery.jplhorizons import Horizons
%load_ext autoreload 
%autoreload 2

In [2]:
fireballs = pd.read_csv('datos/cneos_fireball_data_original.csv', comment='#').sort_values(by=['Calculated Total Impact Energy (kt)'], ascending=False)
fireballs

,Peak Brightness Date/Time (UT),Latitude (deg.),Longitude (deg.),Altitude (km),Velocity (km/s),vx,vy,vz,Total Radiated Energy (J),Calculated Total Impact Energy (kt)
376,2013-02-15 03:20:33,54.8N,61.1E,23.3,18.6,12.8,-13.3,-2.4,3.750000e+14,440.000
178,2018-12-18 23:48:20,56.9N,172.4E,26.0,13.6,6.3,-3.0,-31.2,3.130000e+13,49.000
494,2009-10-08 02:57:00,4.2S,120.6E,19.1,19.2,14.0,-16.0,-6.0,2.000000e+13,33.000
448,2010-12-25 23:24:00,38.0N,158.0E,26.0,18.1,18.0,-2.0,-4.0,2.000000e+13,33.000
954,1994-02-01 22:38:09,2.7N,164.1E,NaN,NaN,NaN,NaN,NaN,1.820000e+13,30.000
...,...,...,...,...,...,...,...,...,...,...
641,2005-04-16 10:40:38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000000e+10,0.073
181,2018-11-15 08:02:44,42N,57W,NaN,NaN,NaN,NaN,NaN,2.000000e+10,0.073
757,2002-03-18 14:44:57,60.4S,120.5W,NaN,NaN,NaN,NaN,NaN,2.000000e+10,0.073
48,2022-03-30 18:19:18,45.9S,171.4W,74.0,NaN,NaN,NaN,NaN,2.000000e+10,0.073


Notemos que el bolido de Chelyabinks es el de mayor energia de impacto con indice 376

In [3]:
indice = 376
meteor = fireballs.loc[indice]
meteor

Peak Brightness Date/Time (UT)         2013-02-15 03:20:33
Latitude (deg.)                                      54.8N
Longitude (deg.)                                     61.1E
Altitude (km)                                         23.3
Velocity (km/s)                                       18.6
vx                                                    12.8
vy                                                   -13.3
vz                                                    -2.4
Total Radiated Energy (J)                375000000000000.0
Calculated Total Impact Energy (kt)                  440.0
Name: 376, dtype: object

In [4]:
def change_coord(x):
    #funcion para trasformar el formato de coordenadas terrestres que da CNEOS
    if x[-1] == 'N' or x[-1] == 'E':
        new = float(x[:-1])
    elif x[-1] == 'S' or x[-1] == 'W':
        new = -float(x[:-1])
    return new    

#Parametros que obtenemos de los datos:
date = meteor['Peak Brightness Date/Time (UT)']
lon = change_coord(meteor['Longitude (deg.)'])
lat = change_coord(meteor['Latitude (deg.)'])
alt = meteor['Altitude (km)']
vx = meteor['vx']
vy = meteor['vy']
vz = meteor['vz']

date, lon, lat, alt, vx, vy, vz

('2013-02-15 03:20:33', 61.1, 54.8, 23.3, 12.8, -13.3, -2.4)

In [5]:
path = 'datos/kernels/'
spy.furnsh([path + 'naif0012.tls', path + 'pck00010.tpc', path + 'earth_fixed.tf', path + 'earth_720101_230601.bpc', path + 'earth_latest_high_prec.bpc'])

In [6]:
r = Geo2Rec(lon, lat, alt)  #en km
r_eclip = Geo2Eclip(lon, lat, alt, date, frame='ITRF93') #en km
print(r, mag(r_eclip))

v = np.array([vx, vy, vz])  #en km/s

T_earth = 86400  
omega = np.array([0,0,(2*np.pi)/T_earth]) 

v_E = (v - spy.vcrss(omega, r)) #km/s
#v_E, -v, mag(v_E), np.arccos((v@r_irtf)/(np.linalg.norm(v)*np.linalg.norm(r_irtf)))*180/np.pi

et = spy.utc2et(date)
mx = spy.pxform('ITRF93', 'J2000', et)
v_eclip = spy.mxv(mx, v_E)

6378.1366 6356.7519
[[-9.64700065e-01  2.63347837e-01  1.30420094e-03]
 [-2.63348099e-01 -9.64700875e-01 -3.00548374e-05]
 [ 1.25024892e-03 -3.72452743e-04  9.99999149e-01]]
[1787.29410694 3237.67768939 5207.6205527 ] 6387.197262180836


### Orbita pre impacto

In [7]:
rb.horizons.SSL_CONTEXT = 'unverified'

AU = 149597870 #km
day = 86400

sim = rb.Simulation()
sim.units = 'km', 's', 'kg'
sim.integrator = "Mercurius"
sim.dt = -86400

sim.add("Sun", hash="Sun", date="JD2456338.639279")
sim.add("399", hash="Earth", date="JD2456338.639279")

sun = sim.particles[0]
r_earth = np.array(sim.particles['Earth'].xyz)
v_earth = np.array(sim.particles['Earth'].vxyz)

r_asteroid = r_eclip + r_earth 
v_asteroid = v_eclip + v_earth #así si es 

print("Position ECLIPJ200: ", r_asteroid/AU)
print("Velocity ECLIPJ200: ", (v_asteroid/AU)*day)

asteroid = sim.add(x=r_asteroid[0], y=r_asteroid[1], z=r_asteroid[2], 
                vx=v_asteroid[0], vy=v_asteroid[1], vz=v_asteroid[2])

Searching NASA Horizons for 'Sun'... 
Found: Sun (10) 
Searching NASA Horizons for '399'... 
Found: Earth (399) 
Position ECLIPJ200:  [-8.23993083e-01  5.43761649e-01 -3.01409733e-05]
Velocity ECLIPJ200:  [-0.01909348 -0.00890474 -0.00137295]


x , y, z (AU) = -8.2296109700e-01 5.4628609900e-01 2.6028354800e-05

vx, vy, vz (AU/day) = -1.7823720100e-02 -9.2681019800e-03 -3.4069700000e-03 

r (AU) = 9.8777197200e-01

In [56]:
sim.particles[1]

<rebound.particle.Particle object at 0x160abd22050, m=5.972365262433514e+24 x=-123266733.4341475 y=81349196.25310567 z=-9717.67125955224 vx=-16.944351995467 vy=-24.94128759172976 vz=0.001486731987759171>

In [ ]:
from astropy.time import Time
t = Time(date, format="iso")
date, t.jd

('2013-02-15 03:20:33', 2456338.639270833)

In [ ]:
2456338.639279.

In [58]:
AU = 149597870 #km
deg = 180/np.pi
sim.move_to_hel()
chely = sim.particles[-1]
earth = sim.particles[1]
asteroid_statev = chely.xyz + chely.vxyz
o_e = earth.orbit()
o = chely.orbit()
orbit_elements = [o.a/AU, o.e, o.inc*deg, o.Omega*deg, o.omega*deg, o.f] 
orbit_elements_e = [o_e.a/AU, o_e.e, o_e.inc*deg, o_e.Omega*deg, o_e.omega*deg, o.f] 

print(f"Elementos orbitales instantaneos chely: ")   
print(f"a={orbit_elements[0]}, e={orbit_elements[1]}, i={orbit_elements[2]}, Omega={np.mod(orbit_elements[3], 360)}, omega={orbit_elements[4]}, f={orbit_elements[5]}")  

print(f"Elementos orbitales instantaneos earth: ")   
print(f"a={orbit_elements_e[0]}, e={orbit_elements_e[1]}, i={orbit_elements_e[2]}, Omega={np.mod(orbit_elements_e[3], 360)}, omega={orbit_elements_e[4]}, f={orbit_elements_e[5]}")  

Elementos orbitales instantaneos chely: 
a=2.8021931481921554, e=0.7487264794451459, i=10.453603435213534, Omega=326.48193338397664, omega=109.16782827636612, f=1.2352735607323089
Elementos orbitales instantaneos earth: 
a=1.0004601886055406, e=0.016810486652860068, i=0.0034264499994486573, Omega=163.15949012729172, omega=301.501435022642, f=1.2352735607323089


In [59]:
deg = 180/np.pi
AU = 149597870 #km
a = 1.73*AU
mu = 1.98847e30*sim.G
#periodo_orb = 2*np.pi*np.sqrt(a**3/mu)

t_end = 4*3.154e+7
print("tiempo de integracion: ", t_end)

N = 10
times = np.linspace(0, t_end, N)

for i,time in enumerate(times):
    sim.integrate(-time)
    sim.move_to_hel()
    chely = sim.particles[-1]
    o = chely.orbit()
    orbit_elements = [o.a/AU, o.e, o.inc*deg, o.Omega*deg, o.omega*deg, o.f] 
    print(f"Elementos orbitales osculantes en t: {-time} s")   
    print(f"a={orbit_elements[0]}, e={orbit_elements[1]}, i={orbit_elements[2]}, Omega={np.mod(orbit_elements[3], 360)}, omega={orbit_elements[4]}, f={orbit_elements[5]}")  

tiempo de integracion:  126160000.0
Elementos orbitales osculantes en t: -0.0 s
a=2.8021931481921554, e=0.7487264794451459, i=10.453603435213534, Omega=326.48193338397664, omega=109.16782827636612, f=1.2352735607323089
Elementos orbitales osculantes en t: -14017777.777777778 s
a=2.6016065394443695, e=0.7271753070981835, i=9.265135650198031, Omega=326.4957321492392, omega=109.29776791131692, f=4.229073795576982
Elementos orbitales osculantes en t: -28035555.555555556 s
a=2.60159828718739, e=0.7271743438887951, i=9.265127510669009, Omega=326.495755787801, omega=109.29781549358215, f=3.7190063041208123
Elementos orbitales osculantes en t: -42053333.333333336 s
a=2.601598035519822, e=0.7271743362167779, i=9.265126859149975, Omega=326.4957604940925, omega=109.29781740492982, f=3.4727909182080516
Elementos orbitales osculantes en t: -56071111.11111111 s
a=2.601597965025254, e=0.7271743502671199, i=9.26512594819299, Omega=326.49577006193255, omega=109.29781133736113, f=3.294407883123029
Eleme

### Integrando por 4 periodos orbitales

In [36]:
rb.horizons.SSL_CONTEXT = 'unverified'

sim = rb.Simulation()
sim.units = 'km', 's', 'kg'
sim.integrator = "WHFast"
sim.dt = -86400

for body in ["Sun", "Earth", "Mercury", "Venus", "Earth", "Mars", "Jupiter", "Saturn", "Uranus", "Neptune"]:
    sim.add(body, hash=body, date=date)

sun = sim.particles[0]
r_earth = np.array(sim.particles['Earth'].xyz)
v_earth = np.array(sim.particles['Earth'].vxyz)

r_asteroid = r_eclip  + r_earth 
v_asteroid = v_eclip + v_earth #así si es 

asteroid = sim.add(x=r_asteroid[0], y=r_asteroid[1], z=r_asteroid[2], 
                vx=v_asteroid[0], vy=v_asteroid[1], vz=v_asteroid[2])

Searching NASA Horizons for 'Sun'... 
Found: Sun (10) 
Searching NASA Horizons for 'Earth'... 
Found: Earth-Moon Barycenter (3) (chosen from query 'Earth')
Searching NASA Horizons for 'Mercury'... 
Found: Mercury Barycenter (199) (chosen from query 'Mercury')
Searching NASA Horizons for 'Venus'... 
Found: Venus Barycenter (299) (chosen from query 'Venus')
Searching NASA Horizons for 'Earth'... 
Found: Earth-Moon Barycenter (3) (chosen from query 'Earth')
Searching NASA Horizons for 'Mars'... 
Found: Mars Barycenter (4) (chosen from query 'Mars')
Searching NASA Horizons for 'Jupiter'... 
Found: Jupiter Barycenter (5) (chosen from query 'Jupiter')
Searching NASA Horizons for 'Saturn'... 
Found: Saturn Barycenter (6) (chosen from query 'Saturn')
Searching NASA Horizons for 'Uranus'... 
Found: Uranus Barycenter (7) (chosen from query 'Uranus')
Searching NASA Horizons for 'Neptune'... 
Found: Neptune Barycenter (8) (chosen from query 'Neptune')


In [37]:
AU = 149597870 #km
day = 86400

print("Position ECLIPJ200: ", r_asteroid/AU)
print("Velocity ECLIPJ200: ", (v_asteroid/AU)*day)

Position ECLIPJ200:  [-8.23985437e-01  5.43762405e-01  1.29035201e-04]
Velocity ECLIPJ200:  [-0.01963751 -0.0097666  -0.00350489]


In [38]:
deg = 180/np.pi
AU = 149597870 #km
a = 1.73*AU
mu = 1.98847e30*sim.G
periodo_orb = 2*np.pi*np.sqrt(a**3/mu)

t_end = 4*periodo_orb 
print("tiempo de integracion: ", t_end)

N = 10
times = np.linspace(0, t_end, N)

orbit_elements = np.zeros((N, 6))
asteroid_statev = np.zeros((N, 6))
for i,time in enumerate(times):
    sim.integrate(-time)
    sim.move_to_com()
    chely = sim.particles[-1]
    asteroid_statev[i] = chely.xyz + chely.vxyz
    o = chely.orbit(primary=sun)
    orbit_elements[i] = [o.a/AU, o.e, o.inc*deg, o.Omega*deg, o.omega*deg, o.f] 

osc = orbit_elements[-1]
print(f"Elementos orbitales osculantes ultimo t: {time} s")   
print(f"a={osc[0]}, e={osc[1]}, i={osc[2]}, Omega={osc[3]}, omega={osc[4]}, f={osc[5]}")  

tiempo de integracion:  287238006.36966944
Elementos orbitales osculantes ultimo t: 287238006.36966944 s
a=2.7731153035651137, e=0.7418651811452246, i=10.355774344339327, Omega=-33.00414668262138, omega=108.6087054833723, f=1.7242701086050847


### Calculando elementos orbitales osculantes con spy

In [39]:
r_int = asteroid_statev[-1]
et = spy.utc2et(date)
state = [r_int[0], r_int[1], r_int[2], r_int[3], r_int[4], r_int[5]]
mu = 1.98847e30*sim.G

osc_spy = spy.oscelt(state, et, mu)
print(f"Elementos orbitales osculantes ultimo t calculados con SPY: {time} s")   
print(f"q={osc_spy[0]/AU}, e={osc_spy[1]}, i={osc_spy[2]*deg}, Omega={osc_spy[3]*deg}, omega={osc_spy[4]*deg}, f={osc_spy[5]}") 

Elementos orbitales osculantes ultimo t calculados con SPY: 287238006.36966944 s
q=0.7095893356534698, e=0.7409334833238133, i=10.362411462908712, Omega=327.03631474234555, omega=108.20196217316364, f=0.2944232216352548


### Comparando datos del profe: 

Projected impact site:
Lat. : 55.07815N
Lon. : 60.09285E
Alt. : 178.4583 m

La fecha de impacto es:
Julian Date = 2456338.639279

La dirección del bolido era:
Radiant:
Az.  : 105.8913
Alt. : 17.84583

Las coordenadas del asteroide en el momento del impacto relativas a ECLIPJ2000 son:
Position vector at impact time (Ecliptic J2000.0):

x , y, z (AU) = -8.2296109700e-01 5.4628609900e-01 2.6028354800e-05
vx, vy, vz (AU/day) = -1.7823720100e-02 -9.2681019800e-03 -3.4069700000e-03 
r (AU) = 9.8777197200e-01

In [40]:
from astropy.time import Time
import spiceypy as spy

jd = 2456338.639279

t = Time(jd, format='jd', scale='utc')
utc_string = t.utc.iso 
et = spy.utc2et(utc_string)
utc_string

'2013-02-15 03:20:33.706'

In [41]:
from astropy.time import Time
import spiceypy as spy

jd = 2456338.639279

t = Time(jd, format='jd', scale='utc')
utc_string = t.utc.iso 
et = spy.utc2et(utc_string)

print(f"UTC time       : {utc_string}")
print(f"Ephemeris Time : {et} seconds since J2000.0")

r = Geo2Rec(lon=60.09285, lat=55.07815, alt=178.4583) #en km
r_eclip = Geo2Eclip(lon=60.09285, lat=55.07815, alt=178.4583, et=et, frame='IAU_EARTH') #return units base on "alt" units

v = np.array([vx, vy, vz])  

T_earth = 2 * np.pi / 86400 + 2 * np.pi / (365.25 * 86400)
#T_earth = 86164.0905
omega = np.array([0,0,T_earth]) 
print(f"Rotation Vector : {omega}")

v_E = (v - spy.vcrss(omega, r))
mx = spy.pxform('IAU_EARTH', 'ECLIPJ2000', et)
v_eclip = spy.mxv(mx, v_E)

UTC time       : 2013-02-15 03:20:33.706
Ephemeris Time : 414170500.8911254 seconds since J2000.0
6378.1366 6356.7519
[[-0.96498118  0.26231601  0.00127587]
 [-0.24018089 -0.88548749  0.39777511]
 [ 0.10547255  0.38353906  0.91748206]]
Rotation Vector : [0.00000000e+00 0.00000000e+00 7.29211543e-05]


In [42]:
rb.horizons.SSL_CONTEXT = 'unverified'

sim = rb.Simulation()
sim.units = 'km', 's', 'kg'
sim.integrator = "WHFast"
#sim.dt = 86400

for body in ["Sun", "Earth", "Mercury", "Venus", "Earth", "Mars", "Jupiter", "Saturn", "Uranus", "Neptune"]:
    sim.add(body, hash=body, date=date)

sun = sim.particles[0]
r_earth = np.array(sim.particles['Earth'].xyz)
v_earth = np.array(sim.particles['Earth'].vxyz)

r_asteroid = r_eclip  + r_earth 
v_asteroid = v_eclip + v_earth #así si es 

asteroid = sim.add(x=r_asteroid[0], y=r_asteroid[1], z=r_asteroid[2], 
                vx=v_asteroid[0], vy=v_asteroid[1], vz=v_asteroid[2])

Searching NASA Horizons for 'Sun'... 
Found: Sun (10) 
Searching NASA Horizons for 'Earth'... 
Found: Earth-Moon Barycenter (3) (chosen from query 'Earth')
Searching NASA Horizons for 'Mercury'... 
Found: Mercury Barycenter (199) (chosen from query 'Mercury')
Searching NASA Horizons for 'Venus'... 
Found: Venus Barycenter (299) (chosen from query 'Venus')
Searching NASA Horizons for 'Earth'... 
Found: Earth-Moon Barycenter (3) (chosen from query 'Earth')
Searching NASA Horizons for 'Mars'... 
Found: Mars Barycenter (4) (chosen from query 'Mars')
Searching NASA Horizons for 'Jupiter'... 
Found: Jupiter Barycenter (5) (chosen from query 'Jupiter')
Searching NASA Horizons for 'Saturn'... 
Found: Saturn Barycenter (6) (chosen from query 'Saturn')
Searching NASA Horizons for 'Uranus'... 
Found: Uranus Barycenter (7) (chosen from query 'Uranus')
Searching NASA Horizons for 'Neptune'... 
Found: Neptune Barycenter (8) (chosen from query 'Neptune')


In [43]:
AU = 149597870 #km
day = 86400

print("Position ECLIPJ200: ", r_asteroid/AU)
print("Velocity ECLIPJ200: ", (v_asteroid/AU)*day)
print("Velocity mag: ", mag(v_E))

Position ECLIPJ200:  [-8.23964890e-01  5.43791877e-01 -2.12928445e-05]
Velocity ECLIPJ200:  [-0.0190926  -0.00988658 -0.00345363]
Velocity mag:  18.875651936295935


In [44]:
deg = 180/np.pi
sim.move_to_hel()
chely = sim.particles[-1]
asteroid_statev = chely.xyz + chely.vxyz
o = chely.orbit()
orbit_elements = [o.a/AU, o.e, o.inc*deg, o.Omega*deg, o.omega*deg, o.f] 

print(f"Elementos orbitales instantaneos: ")   
print(f"a={orbit_elements[0]}, e={orbit_elements[1]}, i={orbit_elements[2]}, Omega={orbit_elements[3]}, omega={orbit_elements[4]}, f={orbit_elements[5]}")  

Elementos orbitales instantaneos: 
a=2.349998623288995, e=0.7002048902389053, i=10.42598501744337, Omega=-33.43037765661752, omega=107.74425201521989, f=1.2612204599706907


In [45]:
sim.status()

---------------------------------
REBOUND version:     	4.4.7
REBOUND built on:    	Mar  9 2025 20:56:14
Number of particles: 	11
Selected integrator: 	whfast
Simulation time:     	0.0000000000000000e+00
Current timestep:    	0.001000
---------------------------------
<rebound.particle.Particle object at 0x160abd36ad0, m=1.9884754159566474e+30 x=0.0 y=0.0 z=0.0 vx=0.0 vy=0.0 vz=0.0>
<rebound.particle.Particle object at 0x160abd34450, m=6.045825576341311e+24 x=-123111951.28375661 y=81726102.64341603 z=-2371.8118567586353 vx=-16.959834981476288 vy=-24.92864501001822 vz=0.0007770261542535619>
<rebound.particle.Particle object at 0x160abd36ad0, m=3.301109449002709e+23 x=19555578.371584143 y=41808991.292543195 z=1621779.9114351934 vx=-53.86316129426408 vy=22.539078024709852 vz=6.7836405815569005>
<rebound.particle.Particle object at 0x160abd34450, m=4.867466257521635e+24 x=57831870.418285474 y=-92197998.23609774 z=-4600984.889498175 vx=29.433165624929565 vy=18.486879224752933 vz=-1.44536625

In [46]:
rb.horizons.SSL_CONTEXT = 'unverified'

sim = rb.Simulation()
sim.units = 'km', 's', 'kg'
sim.integrator = "WHFast"
#sim.dt = 86400

for body in ["Sun", "Earth", "Mercury", "Venus", "Earth", "Mars", "Jupiter", "Saturn", "Uranus", "Neptune"]:
    sim.add(body, hash=body, date=date)

sun = sim.particles[0]
r_earth = np.array(sim.particles['Earth'].xyz)
v_earth = np.array(sim.particles['Earth'].vxyz)

r_asteroid = r_eclip  + r_earth 
v_asteroid = v_eclip + v_earth #así si es 

asteroid = sim.add(x=r_asteroid[0], y=r_asteroid[1], z=r_asteroid[2], 
                vx=v_asteroid[0], vy=v_asteroid[1], vz=v_asteroid[2])

hora = 3600

#t_end = 4*periodo_orb
t_end = 4*hora
print("tiempo de integracion: ", t_end)

N = 10
times = np.linspace(0, t_end, N)

orbit_elements = np.zeros((N, 6))
for i,time in enumerate(times):
    sim.integrate(-time)
    sim.move_to_com()
    o = sim.particles[-1].orbit(primary=sim.particles[0])
    int_elements = [o.a/AU, o.e, o.inc*deg, o.Omega*deg, o.omega*deg, o.f] 
    orbit_elements[i] = int_elements 

    print(f"Elementos orbitales en tiempo: {time} s")   
    print(f"a={int_elements[0]}, e={int_elements[1]}, i={int_elements[2]}, Omega={np.mod(int_elements[3], 360)}, omega={int_elements[4]}, f={int_elements[5]}")  

Searching NASA Horizons for 'Sun'... 
Found: Sun (10) 
Searching NASA Horizons for 'Earth'... 
Found: Earth-Moon Barycenter (3) (chosen from query 'Earth')
Searching NASA Horizons for 'Mercury'... 
Found: Mercury Barycenter (199) (chosen from query 'Mercury')
Searching NASA Horizons for 'Venus'... 
Found: Venus Barycenter (299) (chosen from query 'Venus')
Searching NASA Horizons for 'Earth'... 
Found: Earth-Moon Barycenter (3) (chosen from query 'Earth')
Searching NASA Horizons for 'Mars'... 
Found: Mars Barycenter (4) (chosen from query 'Mars')
Searching NASA Horizons for 'Jupiter'... 
Found: Jupiter Barycenter (5) (chosen from query 'Jupiter')
Searching NASA Horizons for 'Saturn'... 
Found: Saturn Barycenter (6) (chosen from query 'Saturn')
Searching NASA Horizons for 'Uranus'... 
Found: Uranus Barycenter (7) (chosen from query 'Uranus')
Searching NASA Horizons for 'Neptune'... 
Found: Neptune Barycenter (8) (chosen from query 'Neptune')
tiempo de integracion:  14400
Elementos orbita

In [47]:
rb.horizons.SSL_CONTEXT = 'unverified'

sim = rb.Simulation()
sim.units = 'km', 's', 'kg'
sim.integrator = "WHFast"
sim.dt = -86400

for body in ["Sun", "Earth", "Mercury", "Venus", "Earth", "Mars", "Jupiter", "Saturn", "Uranus", "Neptune"]:
    sim.add(body, hash=body, date=date)

sun = sim.particles[0]
r_earth = np.array(sim.particles['Earth'].xyz)
v_earth = np.array(sim.particles['Earth'].vxyz)

r_asteroid = r_eclip  + r_earth 
v_asteroid = v_eclip + v_earth #así si es 

asteroid = sim.add(x=r_asteroid[0], y=r_asteroid[1], z=r_asteroid[2], 
                vx=v_asteroid[0], vy=v_asteroid[1], vz=v_asteroid[2])

hora = 3600
deg = 180/np.pi
AU = 149597870 #km
a = 1.73*AU
mu = 1.98847e30*sim.G
periodo_orb = 2*np.pi*np.sqrt(a**3/mu)

print("Position ECLIPJ200: ", r_asteroid/AU)
print("Velocity ECLIPJ200: ", (v_asteroid/AU)*day)
print("Velocity mag: ", mag(v_E))

t_end = 4*periodo_orb
#t_end = 24*hora
print("tiempo de integracion: ", t_end)

N = 10
times = np.linspace(0, t_end, N)

asteroid_statev = np.zeros((N, 6))
for i,time in enumerate(times):
    sim.integrate(-time)
    sim.move_to_com()
    asteroid_statev[i] = sim.particles[-1].xyz + sim.particles[-1].vxyz 

Searching NASA Horizons for 'Sun'... 
Found: Sun (10) 
Searching NASA Horizons for 'Earth'... 
Found: Earth-Moon Barycenter (3) (chosen from query 'Earth')
Searching NASA Horizons for 'Mercury'... 
Found: Mercury Barycenter (199) (chosen from query 'Mercury')
Searching NASA Horizons for 'Venus'... 
Found: Venus Barycenter (299) (chosen from query 'Venus')
Searching NASA Horizons for 'Earth'... 
Found: Earth-Moon Barycenter (3) (chosen from query 'Earth')
Searching NASA Horizons for 'Mars'... 
Found: Mars Barycenter (4) (chosen from query 'Mars')
Searching NASA Horizons for 'Jupiter'... 
Found: Jupiter Barycenter (5) (chosen from query 'Jupiter')
Searching NASA Horizons for 'Saturn'... 
Found: Saturn Barycenter (6) (chosen from query 'Saturn')
Searching NASA Horizons for 'Uranus'... 
Found: Uranus Barycenter (7) (chosen from query 'Uranus')
Searching NASA Horizons for 'Neptune'... 
Found: Neptune Barycenter (8) (chosen from query 'Neptune')
Position ECLIPJ200:  [-8.23964890e-01  5.43791

In [48]:
AU = 149597870 #km
deg = 180/np.pi
for element in asteroid_statev:
    r_int = element
    et = spy.utc2et(date)
    state = [r_int[0], r_int[1], r_int[2], r_int[3], r_int[4], r_int[5]]

    osc_spy = spy.oscelt(state, et, mu)
    print(f"Elementos orbitales osculantes ultimo t calculados con SPY: {time} s")   
    print(f"q={osc_spy[0]/AU}, e={osc_spy[1]}, i={osc_spy[2]*deg}, Omega={osc_spy[3]*deg}, omega={osc_spy[4]*deg}, f={osc_spy[5]}") 

Elementos orbitales osculantes ultimo t calculados con SPY: 287238006.36966944 s
q=0.7049055860225364, e=0.7015602118425917, i=10.425985017443395, Omega=326.5696223433825, omega=107.84907742242555, f=0.20066526281148883
Elementos orbitales osculantes ultimo t calculados con SPY: 287238006.36966944 s
q=0.7092513745799958, e=0.697340073766127, i=10.354375289770184, Omega=326.371254611087, omega=108.19768618765555, f=4.714480597244144
Elementos orbitales osculantes ultimo t calculados con SPY: 287238006.36966944 s
q=0.711603165009859, e=0.6964497236461668, i=10.349431362017441, Omega=326.4180278712254, omega=108.15120584015779, f=2.9420353931492667
Elementos orbitales osculantes ultimo t calculados con SPY: 287238006.36966944 s
q=0.7147627258932848, e=0.6955154589902554, i=10.349089895944825, Omega=326.4410170770551, omega=108.23393617769547, f=1.1698503496409525
Elementos orbitales osculantes ultimo t calculados con SPY: 287238006.36966944 s
q=0.7078750132398696, e=0.6958441733727445, i=